In [1]:
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [2]:
s3 = boto3.client('s3')

bucket_name = 'pract-python-postgres-pipleline-csv-file'
s3_key = 'brownze/bronze_orders_raw.csv'
local_file = 'bronze_orders_raw.csv'

s3.download_file(bucket_name, s3_key, local_file)

print("s3 file downloaded successfully")

s3 file downloaded successfully


In [3]:
spark = SparkSession.builder.appName('Practice').getOrCreate()

print(spark.version)

4.1.2


In [4]:
df = spark.read.csv("bronze_orders_raw.csv", header=True)

df.show()

+--------+-----------+---------+--------+-----------+--------+------+----------+
|order_id|customer_id|     city| product|   category|quantity|amount|order_date|
+--------+-----------+---------+--------+-----------+--------+------+----------+
|       1|      C1001|   Mumbai|   Phone|Electronics|       3| 32598|2025-04-25|
|       2|      C1002|Hyderabad|   Chair|  Furniture|       1| 71982|2025-02-14|
|       3|      C1003|   Mumbai| Monitor|Electronics|       1| 12780|2025-04-22|
|       4|      C1004|  Chennai|    Desk|  Furniture|       5|  3978|2025-10-15|
|       5|      C1005|Hyderabad|    Desk|  Furniture|       5| 55487|2025-04-23|
|       6|      C1006|  Chennai|  Tablet|Electronics|       3| -2500|2026-01-24|
|       7|      C1007|Hyderabad|   Chair|  Furniture|       4| 45097|2025-05-23|
|       8|      C1008|     Pune|   Chair|  Furniture|       3| 13896|2025-02-17|
|       9|      C1009|   Mumbai| Monitor|Electronics|       3| 45582|2025-11-06|
|      10|      C1010|   Mum

In [5]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_date: string (nullable = true)



In [6]:
"""column_name = ['order_id', 'quantity', 'amount']

for column in column_name:
    df = df.withColumn(column, col(column).cast("int"))

df = df.withColumn("order_date", col("order_date").cast("date"))

df.printSchema()"""

'column_name = [\'order_id\', \'quantity\', \'amount\']\n\nfor column in column_name:\n    df = df.withColumn(column, col(column).cast("int"))\n\ndf = df.withColumn("order_date", col("order_date").cast("date"))\n\ndf.printSchema()'

In [7]:
df = df.dropDuplicates()

In [8]:
df.show()

+--------+-----------+---------+--------+-----------+--------+------+----------+
|order_id|customer_id|     city| product|   category|quantity|amount|order_date|
+--------+-----------+---------+--------+-----------+--------+------+----------+
|      82|      C1082|    Delhi|  Laptop|Electronics|       2| 33206|2025-12-08|
|      74|      C1074|  Chennai|   Mouse|Electronics|       2| 45442|2025-04-15|
|     117|      C1117|Hyderabad|    Desk|  Furniture|       3| 68827|2025-09-06|
|     122|      C1122|    Delhi| Monitor|Electronics|       5| 61569|2025-08-01|
|     150|      C1150|     Pune|Keyboard|Electronics|       1| 37059|2026-03-28|
|      84|      C1084|     Pune|   Chair|  Furniture|       2| 23706|2026-03-28|
|      32|      C1032|Hyderabad|   Phone|Electronics|       3| 67040|2025-11-08|
|      97|      C1097|    Delhi|    Desk|  Furniture|       4|  9671|2026-05-01|
|      88|      C1088|Bangalore|    Desk|  Furniture|       3| 40501|2026-02-25|
|     102|      C1102|Bangal

In [9]:
df.count()

150

In [10]:
df = df.na.drop()

In [11]:
df.count()

146

In [12]:
df = df.withColumn("amount", col("amount").cast("int"))

df = df.filter(col("amount")>0)

In [13]:
df.count()

144

In [14]:
df.show()

+--------+-----------+---------+--------+-----------+--------+------+----------+
|order_id|customer_id|     city| product|   category|quantity|amount|order_date|
+--------+-----------+---------+--------+-----------+--------+------+----------+
|      82|      C1082|    Delhi|  Laptop|Electronics|       2| 33206|2025-12-08|
|      74|      C1074|  Chennai|   Mouse|Electronics|       2| 45442|2025-04-15|
|     117|      C1117|Hyderabad|    Desk|  Furniture|       3| 68827|2025-09-06|
|     122|      C1122|    Delhi| Monitor|Electronics|       5| 61569|2025-08-01|
|     150|      C1150|     Pune|Keyboard|Electronics|       1| 37059|2026-03-28|
|      84|      C1084|     Pune|   Chair|  Furniture|       2| 23706|2026-03-28|
|      32|      C1032|Hyderabad|   Phone|Electronics|       3| 67040|2025-11-08|
|      97|      C1097|    Delhi|    Desk|  Furniture|       4|  9671|2026-05-01|
|      88|      C1088|Bangalore|    Desk|  Furniture|       3| 40501|2026-02-25|
|     102|      C1102|Bangal

In [15]:
df.write.mode("overwrite").parquet("silver/orders")